<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
# Leakage sanity check on the expanded feature set — same method as w03
new_features_to_check = [
    "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions"
]
corrs = df[new_features_to_check].corrwith(df["trend_pct"]).sort_values(key=abs, ascending=False)
print("Correlation of expanded features with trend_pct (label source):")
print(corrs.round(3))

Correlation of expanded features with trend_pct (label source):
days_with_impressions   -0.032
days_with_sessions      -0.008
scroll_events_90d       -0.005
users_90d               -0.004
ai_sessions_90d         -0.003
sessions_90d            -0.003
clicks_90d              -0.002
engaged_sessions_90d     0.002
pageviews_90d           -0.001
dtype: float64


All expanded features show negligible correlation with trend_pct (max magnitude 0.032), consistent with the clean feature check in w03_feature_leakage_check. This confirms the model's Precision@50 improvement reflects real signal, not label leakage.

In [19]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/MuhammadJawadFasih/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

# Clone your repository if it is not already present
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into your repository
os.chdir(REPO_DIR)

print("Working directory:")
print(os.getcwd())

# Check that the dataset exists
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

print("\nDataset exists:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    print("Dataset found:", DATA_PATH)
else:
    print("Dataset NOT found.")

Working directory:
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship

Dataset exists: True
Dataset found: data/raw/content_refresh_anonymized.csv


In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will model whether a page is declining using a binary classification approach. I will start with Logistic Regression as a simple, readable learned model and compare it with a Random Forest that can capture nonlinear relationships between the search and engagement signals.

The model output will be the predicted probability of decline. I will use that probability as a ranking score because the capstone question is not only “is this page declining?” but also “which pages should be reviewed first?”

The target label is derived from trend_direction == "down". I will not use trend_direction or trend_pct as model features because doing so would leak the outcome into the model.

The models are decision-support tools. Their predictions show patterns observed in this dataset and do not prove that refreshing a page will improve search performance.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Binary outcome
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Rows:", len(df))
print("Declining pages:", df["is_declining_label"].sum())
print("Decline base rate:", round(df["is_declining_label"].mean(), 4))

Rows: 30000
Declining pages: 16262
Decline base rate: 0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped train/test split based on client_id. All pages from a client are kept together, so a client that appears in the training set cannot also appear in the test set.

This is more honest than a random row split because pages from the same client may share content, traffic, or search characteristics. Allowing the same client into both sets could make test performance look better than it would be on unseen clients.

I use a fixed random seed of 42 so the split can be reproduced.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Features deliberately exclude:
# - trend_direction
# - trend_pct
# - client_id
# - content_id
# - fields directly derived from future outcome

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

model_df = df[
    ["content_id", "client_id", "is_declining_label"] + feature_columns
].copy()

# Replace invalid numeric values
model_df[feature_columns] = (
    model_df[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

# Median fill for missing numeric values
for col in feature_columns:
    model_df[col] = model_df[col].fillna(
        model_df[col].median()
    )

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

X_train = train[feature_columns]
y_train = train["is_declining_label"]

X_test = test[feature_columns]
y_test = test["is_declining_label"]

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])

print("Client overlap:", len(overlap))
print("Train decline rate:", round(y_train.mean(), 4))
print("Test decline rate:", round(y_test.mean(), 4))

assert len(overlap) == 0

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0
Train decline rate: 0.5501
Test decline rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare the Week-4 baseline with Logistic Regression and Random Forest on exactly the same held-out clients.

The primary ranking metric is Precision@50: among the 50 pages ranked highest for review, what fraction are actually labeled as declining? I also report Precision@20 to check whether the result holds at a smaller review budget.

Higher Precision@K means that a content team reviewing only the highest-ranked pages would encounter a larger proportion of observed declining pages.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# Precision@K helper
# --------------------------------------------------

def precision_at_k(y_true, scores, k):
    result = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = result.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y_true"].mean()


# --------------------------------------------------
# WEEK-4 BASELINE RECREATED ON THE TEST SET
# --------------------------------------------------

baseline_test = test.copy()

baseline_test["position_bucket"] = pd.cut(
    baseline_test["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# IMPORTANT:
# thresholds are learned from TRAINING DATA only

train_for_baseline = train.copy()

train_for_baseline["position_bucket"] = pd.cut(
    train_for_baseline["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

volume_threshold = train_for_baseline[
    "search_volume"
].median()

position_ctr_median = (
    train_for_baseline[
        train_for_baseline["avg_position"] > 0
    ]
    .groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .median()
)

baseline_test["position_ctr_median"] = (
    baseline_test["position_bucket"]
    .map(position_ctr_median)
    .astype(float)
)

baseline_test["high_volume"] = (
    baseline_test["search_volume"]
    >= volume_threshold
)

baseline_test["low_ctr_vs_position"] = (
    (baseline_test["avg_position"] > 0)
    & baseline_test["position_ctr_median"].notna()
    & (
        baseline_test["ctr"]
        <= baseline_test["position_ctr_median"]
    )
)

baseline_test["baseline_score"] = np.where(
    baseline_test["high_volume"]
    & baseline_test["low_ctr_vs_position"],
    baseline_test["search_volume"],
    0
)


# --------------------------------------------------
# LOGISTIC REGRESSION
# --------------------------------------------------

logistic_model = Pipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

logistic_scores = logistic_model.predict_proba(
    X_test
)[:, 1]


# --------------------------------------------------
# RANDOM FOREST
# --------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_scores = rf_model.predict_proba(
    X_test
)[:, 1]


# --------------------------------------------------
# SAME METRICS, SAME TEST SET
# --------------------------------------------------

base_rate = y_test.mean()

results = pd.DataFrame({
    "method": [
        "Base rate",
        "Week-4 baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "precision_at_20": [
        base_rate,
        precision_at_k(
            y_test,
            baseline_test["baseline_score"],
            20
        ),
        precision_at_k(
            y_test,
            logistic_scores,
            20
        ),
        precision_at_k(
            y_test,
            rf_scores,
            20
        )
    ],
    "precision_at_50": [
        base_rate,
        precision_at_k(
            y_test,
            baseline_test["baseline_score"],
            50
        ),
        precision_at_k(
            y_test,
            logistic_scores,
            50
        ),
        precision_at_k(
            y_test,
            rf_scores,
            50
        )
    ]
})

print(
    results.round(3).to_string(index=False)
)

             method  precision_at_20  precision_at_50
          Base rate            0.511            0.511
    Week-4 baseline            0.550            0.500
Logistic Regression            0.750            0.720
      Random Forest            0.650            0.620


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*         
Note: this test set covers only 7 held-out clients (vs. 25 in training), so Precision@50 here is a small-sample estimate and could shift meaningfully with a different client split.

The Logistic Regression model achieved the strongest Precision@50 among the learned models at 0.720. Its largest absolute coefficients were associated with users_90d, sessions_90d, and days_with_impressions, making these the strongest model signals by coefficient magnitude. The model still produced both false positives and false negatives, so its ranking should be used to prioritize review rather than treated as a definitive prediction of decline. These relationships are observed in this dataset and should not be interpreted as causal effects.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 4 — ERRORS AND INTERPRETATION
# ============================================================

# Compare the two learned models using Precision@50
model_scores = {
    "Logistic Regression": precision_at_k(
        y_test, logistic_scores, 50
    ),
    "Random Forest": precision_at_k(
        y_test, rf_scores, 50
    )
}

best_model_name = max(
    model_scores,
    key=model_scores.get
)

if best_model_name == "Random Forest":
    best_scores = rf_scores
    best_model = rf_model
else:
    best_scores = logistic_scores
    best_model = logistic_model

print("Best learned model:", best_model_name)
print(
    "Precision@50:",
    round(model_scores[best_model_name], 3)
)


# ------------------------------------------------------------
# Create prediction/error table
# ------------------------------------------------------------

error_df = test[
    ["content_id", "client_id"] + feature_columns
].copy()

error_df["actual_decline"] = y_test.values
error_df["predicted_probability"] = best_scores

# Classification threshold for error analysis only
error_df["predicted_decline"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)

error_df["error_type"] = "Correct"

error_df.loc[
    (error_df["actual_decline"] == 0)
    & (error_df["predicted_decline"] == 1),
    "error_type"
] = "False Positive"

error_df.loc[
    (error_df["actual_decline"] == 1)
    & (error_df["predicted_decline"] == 0),
    "error_type"
] = "False Negative"


# ------------------------------------------------------------
# Error counts
# ------------------------------------------------------------

print("\nError counts:")
print(
    error_df["error_type"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

false_positives = (
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
)

print("\nTop 3 false positives:")
print(
    false_positives[
        [
            "content_id",
            "actual_decline",
            "predicted_probability",
            "search_volume",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ]
    .head(3)
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

false_negatives = (
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
)

print("\nTop 3 false negatives:")
print(
    false_negatives[
        [
            "content_id",
            "actual_decline",
            "predicted_probability",
            "search_volume",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ]
    .head(3)
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

if best_model_name == "Random Forest":

    importance = pd.Series(
        best_model.feature_importances_,
        index=feature_columns
    ).sort_values(ascending=False)

else:

    coefficients = best_model.named_steps["model"].coef_[0]

    importance = pd.Series(
        np.abs(coefficients),
        index=feature_columns
    ).sort_values(ascending=False)


print("\nTop 10 model signals:")
print(
    importance.head(10)
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# Compact interpretation
# ------------------------------------------------------------

print("\nInterpretation:")
print(
    f"The strongest learned model was {best_model_name}, "
    f"with Precision@50 = "
    f"{model_scores[best_model_name]:.3f}."
)

print(
    "The most influential signals were:"
)

for feature, value in importance.head(3).items():
    print(f"- {feature}: {value:.4f}")

print(
    "\nError analysis shows that the model does not perfectly "
    "separate declining from non-declining pages. False positives "
    "represent pages the model ranked as higher-risk even though "
    "they were not labeled as declining, while false negatives "
    "represent observed declining pages that the model did not "
    "rank highly enough."
)

print(
    "\nThese are observed patterns in the evaluation data and "
    "should be interpreted as decision-support signals rather "
    "than causal effects."
)

Best learned model: Logistic Regression
Precision@50: 0.72

Error counts:
error_type
Correct           3569
False Negative    1306
False Positive    1288

Top 3 false positives:
          content_id  actual_decline  predicted_probability  search_volume  ctr  avg_position  days_since_last_update
content_374e795aab68               0                  0.867          880.0 0.85          31.0                      20
content_c94a53e3bfb8               0                  0.866            0.0 0.23           8.1                      20
content_26d48a980581               0                  0.865            0.0 0.00           4.6                     106

Top 3 false negatives:
          content_id  actual_decline  predicted_probability  search_volume  ctr  avg_position  days_since_last_update
content_2a0400a50c04               1                  0.500          260.0 0.14           9.9                      25
content_bdf67436ad54               1                  0.499           20.0 0.10          1

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.